In [0]:
# Gold layer: bridge_supplier_products (Kimball supplier-product
#         pricing bridge) - closes the gap caught in the gold audit:
#         KPI 4 (Gross Margin Analysis) and KPI 14
#         (Supplier-Product Performance) need supplier cost
#         (supplier_price), which lives only in
#         b2b_ecommerce.silver.supplier_products - until now no
#         gold table carried it (dim_suppliers was deliberately
#         identity-only, with the note 'as-of pricing lives in the
#         supplier_products bridge', but the bridge was never
#         built). This table closes that gap so both KPI views run
#         pure-gold.
# Source: b2b_ecommerce.silver.supplier_products - one row per
#         (supplier_id, product_id), latest-state snapshot
#         ("upsert merge - latest updated_at wins, no history" is
#         silver's stated design).
# Grain: one row per (supplier_id, product_id) - the composite
#         business key, MERGE key, and the join key KPI 4/14 views
#         use (fact_order_items carries supplier_sk + product_sk;
#         views resolve business keys through dim_suppliers /
#         dim_products, then join here on the pair).
# Why SCD2 on supplier_price (same decision as dim_products): cost
#         is a pricing attribute KPI 4 must compare AS OF the order
#         date once cost changes ever materialize. Silver holds one
#         state per pair today, so the first build is a single open
#         version per pair; a later cost change closes the old
#         window and opens a new one. Margin views then use the
#         fact_order_items as-of pattern (order created_at between
#         __START_AT and __END_AT, falling back to the pair's FIRST
#         version when the order predates the bridge - every
#         current order does, all bridge versions start at build
#         time, exactly like dim_companies/dim_products did).
# Columns: supplier_id + product_id + supplier_price - the
#         KPI-trimmed set. supplier_sku / min_order_qty /
#         lead_time_days / is_preferred / valid_from / created_at /
#         updated_at excluded (no KPI uses them; gold timestamp
#         policy - updated_at is the silver CDC sequence only,
#         mixed-clock source limitation).
# No surrogate key: unlike the dims there is no single natural key
#         and no fact carries a bridge SK - views join on the
#         composite business key, so a crc32 SK would be dead
#         weight (deliberate, documented divergence from the dim
#         convention; also avoids inventing a concat key the way
#         fact_web_sessions had to avoid crc32 collisions).
# MERGE / idempotency: AUTO CDC FROM SNAPSHOT into a Delta-backed
#         streaming table keyed on (supplier_id, product_id) with
#         history tracked on supplier_price - an identical snapshot
#         diffs to zero events, so retries and no-op runs change
#         nothing; a pair removed from silver is deleted here too
#         (snapshot diff semantics).
# Pre-flight (2026-11-19, verified): 313 rows = 313 distinct pairs,
#         0 null keys, 0 null / non-positive supplier_price, 0
#         supplier FK gaps vs dim_suppliers, 0 product FK gaps vs
#         dim_products, 0 sold (supplier, product) pairs missing
#         from the bridge (100% coverage of fact_order_items), 8
#         below-cost products confirmed (known data issue,
#         retained - KPI 4 surfaces the negative margins).
# Quality: no expectations - silver owns row-level validation, and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT
#         flows anyway (same decision as every other gold table).
# Layout: not partitioned (bridge, not date-grained),
#         liquid-clustered by the composite key (dims cluster by
#         their join key; at 313 rows this is convention, not
#         need).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_SUPPLIER_PRODUCTS = "b2b_ecommerce.silver.supplier_products"


# Silver's complete current-state pricing snapshot. A batch temporary
# view: the engine reads it at each pipeline run (ingestion-time
# snapshot versioning). The select doubles as the column trim; the
# non-null key guard is defensive (AUTO CDC keys must be non-null;
# pre-flight shows 0 nulls today, but the guard stays like the other
# dims).
@dp.temporary_view(name="supplier_products_snapshot")
def supplier_products_snapshot():
    return (
        spark.read.table(SILVER_SUPPLIER_PRODUCTS)
        .where(F.col("supplier_id").isNotNull() & F.col("product_id").isNotNull())
        .select("supplier_id", "product_id", "supplier_price")
    )


# Delta-backed streaming table: the AUTO CDC target, liquid-clustered
# on the composite business key (the join key the KPI views use).
dp.create_streaming_table(
    name="bridge_supplier_products",
    comment=(
        "Kimball supplier-product pricing bridge, SCD Type 2 on supplier_price: one "
        "row version per (supplier_id, product_id), MERGE-loaded from the "
        "b2b_ecommerce.silver.supplier_products current-state snapshot via AUTO CDC "
        "FROM SNAPSHOT keyed on the composite - identical snapshots diff to zero "
        "events, so retries are idempotent; pairs removed from silver are deleted "
        "here. Carries the supplier cost that KPI 4 (gross margin per "
        "product/supplier) and KPI 14 (supplier-product performance) compare against "
        "actual selling price; SCD2 so future cost changes version and margin views "
        "can resolve cost AS OF order created_at (falling back to the pair's first "
        "version when the order predates the bridge). Column set KPI-trimmed to "
        "supplier_price; no surrogate key (views join on the composite business "
        "key, no fact carries a bridge SK)."
    ),
    cluster_by=["supplier_id", "product_id"],
)


# The auto-merge load: diff each snapshot against the last committed one
# and apply inserts/updates/deletes; only a supplier_price change opens
# a new version window (SCD2 on the cost, Type 1 on everything else).
dp.create_auto_cdc_from_snapshot_flow(
    target="bridge_supplier_products",
    source="supplier_products_snapshot",  # ingestion-time snapshot versioning
    keys=["supplier_id", "product_id"],
    stored_as_scd_type=2,
    track_history_column_list=["supplier_price"],
)